# Silver — ticker

`bronze.trusts` + `bronze.yf_pull_log` + `silver.monthly_performance` →
**`silver.ticker`**.

One row per instrument: **all 118 trusts in the metadata plus SPY, IVV and VOO**. The
trusts Yahoo erased get a row too, marked `no-data`. They produce no return rows and enter
no beat-rate denominator, so no published number changes — but the dimension does not
quietly agree with Yahoo that those trusts never existed, which is the bias this project
is about.

Flat, no SCD2: version history is built in Gold by MERGE. Run this **after**
`build_monthly_performance`, whose output supplies the coverage facts.

Expected: **121 rows**.

In [0]:
CREATE TABLE IF NOT EXISTS `index-vs-trust-pipeline`.silver.ticker (
  ticker            STRING  COMMENT 'Business key',
  trust_name        STRING,
  entity_type       STRING  COMMENT 'Trust or Index, so Gold compares them by self-join',
  aic_sector        STRING,
  manager           STRING  COMMENT 'Full comma-separated list, nulls carried not defaulted',
  management_group  STRING,
  manager_structure STRING  COMMENT 'sole or multi, from the count of named managers',
  currency          STRING,
  price_source      STRING  COMMENT 'yahoo, archive or none',
  first_month       INT,
  last_month        INT,
  months_available  INT,
  status            STRING  COMMENT 'Derived from coverage, never asserted',
  data_status       STRING  COMMENT 'usable, stub or no-data'
)
COMMENT 'Every trust and index in the universe, including those with no prices';

In [0]:
CREATE OR REPLACE TEMP VIEW silver_stage_ticker AS
WITH latest AS (
  SELECT MAX(month_key) AS latest_month
  FROM `index-vs-trust-pipeline`.silver.monthly_performance
),
coverage AS (
  SELECT ticker,
         MIN(month_key)    AS first_month,
         MAX(month_key)    AS last_month,
         COUNT(*)          AS months_available,
         MAX(price_source) AS price_source,
         MAX(currency)     AS currency
  FROM `index-vs-trust-pipeline`.silver.monthly_performance
  GROUP BY ticker
),
pull AS (
  SELECT source_ticker, MAX(status) AS pull_status
  FROM `index-vs-trust-pipeline`.bronze.yf_pull_log
  GROUP BY source_ticker
),
meta AS (
  -- Two rows carry a blank ticker (Island Innovation, Witan) and cannot be keyed.
  SELECT ticker, trust_name, aic_sector, manager, management_group
  FROM `index-vs-trust-pipeline`.bronze.trusts
  WHERE ticker IS NOT NULL AND TRIM(ticker) <> ''
),
trusts AS (
  SELECT m.ticker,
         m.trust_name,
         'Trust'                          AS entity_type,
         m.aic_sector,
         m.manager,
         m.management_group,
         -- A scalar expression, not a window function. Buys a free insight: do
         -- solo-managed trusts beat the index more often than committees do?
         CASE WHEN m.manager IS NULL OR TRIM(m.manager) = '' THEN NULL
              WHEN SIZE(SPLIT(m.manager, ', ')) > 1          THEN 'multi'
              ELSE 'sole' END             AS manager_structure,
         c.currency,
         COALESCE(c.price_source, 'none') AS price_source,
         c.first_month,
         c.last_month,
         COALESCE(c.months_available, 0)  AS months_available
  FROM meta m
  LEFT JOIN coverage c ON c.ticker = m.ticker
),
index_rows AS (
  SELECT c.ticker,
         c.ticker             AS trust_name,
         'Index'              AS entity_type,
         CAST(NULL AS STRING) AS aic_sector,
         CAST(NULL AS STRING) AS manager,
         CAST(NULL AS STRING) AS management_group,
         CAST(NULL AS STRING) AS manager_structure,
         c.currency, c.price_source, c.first_month, c.last_month, c.months_available
  FROM coverage c
  WHERE c.ticker IN ('SPY', 'IVV', 'VOO')
),
combined AS (
  SELECT * FROM trusts
  UNION ALL
  SELECT * FROM index_rows
)
SELECT combined.*,
       -- Derived, never asserted. The seed file claimed an is_active column that
       -- contradicted the prices, which is why that file was dropped.
       CASE WHEN combined.last_month = latest.latest_month THEN 'active'
            WHEN combined.last_month IS NOT NULL           THEN 'delisted'
            WHEN pull.pull_status = 'NODATA'               THEN 'delisted'
            ELSE 'active' END AS status,
       -- The 36-month floor is a Gold rule; Silver only labels who would fail it.
       CASE WHEN combined.months_available = 0  THEN 'no-data'
            WHEN combined.months_available < 36 THEN 'stub'
            ELSE 'usable' END AS data_status
FROM combined
CROSS JOIN latest
LEFT JOIN pull ON pull.source_ticker = combined.ticker;

In [0]:
MERGE INTO `index-vs-trust-pipeline`.silver.ticker AS t
USING silver_stage_ticker AS s
   ON t.ticker = s.ticker
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

## Verification

In [0]:
SELECT COUNT(*)                                                        AS rows_total,
       SUM(CASE WHEN entity_type = 'Trust'      THEN 1 ELSE 0 END)     AS trusts,
       SUM(CASE WHEN data_status = 'usable'     THEN 1 ELSE 0 END)     AS usable,
       SUM(CASE WHEN data_status = 'stub'       THEN 1 ELSE 0 END)     AS stub,
       SUM(CASE WHEN data_status = 'no-data'    THEN 1 ELSE 0 END)     AS no_data,
       SUM(CASE WHEN price_source = 'archive'   THEN 1 ELSE 0 END)     AS archive,
       SUM(CASE WHEN status = 'delisted'        THEN 1 ELSE 0 END)     AS delisted
FROM `index-vs-trust-pipeline`.silver.ticker;

Expect **121 / 118 / 101 / 3 / 17 / 2 / 19**.

Two cross-checks worth naming out loud:

- **usable 101 includes the 3 index tickers, so 98 trusts are usable** — the 96 the design
  documented, plus `BCPT` and `CSH` recovered from the archive.
- **delisted 19** = the 16 Yahoo erased, plus `BCPT` and `CSH` (which stopped being priced),
  plus `ADIG` (last bar 2026-03 while everything else runs to 2026-08).

`no-data` is **17**, not 16: `MNTN` joins the erased trusts because its only Yahoo bar sits
in the excluded partial month.

In [0]:
-- Nulls are carried, never defaulted. Substituting 'Unknown' would invent a management
-- group that dim_ticker would then open an SCD2 version on.
SELECT SUM(CASE WHEN manager_structure = 'multi' THEN 1 ELSE 0 END) AS multi_manager,
       SUM(CASE WHEN manager_structure = 'sole'  THEN 1 ELSE 0 END) AS sole_manager,
       SUM(CASE WHEN manager_structure IS NULL
                 AND entity_type = 'Trust'       THEN 1 ELSE 0 END) AS no_manager,
       SUM(CASE WHEN data_status <> 'no-data'
                 AND manager IS NULL             THEN 1 ELSE 0 END) AS priced_but_no_manager,
       COUNT(DISTINCT management_group)                             AS management_groups
FROM `index-vs-trust-pipeline`.silver.ticker;

Expect **70 / 27 / 21 / 3 / 52**.

70 + 27 = 97, the trusts that have manager data at all. Of the 21 without, only **3** are
trusts we actually price — that is the finding EDA recorded as 1.4, and the null is carried
as a null.

In [0]:
-- The survivorship evidence, readable in one query: the trusts that vanished.
SELECT ticker, trust_name, management_group, status, data_status, price_source,
       first_month, last_month, months_available
FROM `index-vs-trust-pipeline`.silver.ticker
WHERE status = 'delisted'
ORDER BY months_available DESC, ticker;

Expect **19 rows**. `BCPT` (158 months) and `CSH` (112) at the top with real history, then
`ADIG`, then the 16 Yahoo returns nothing for — named, with their management groups, and
counted.

This is the table to put on screen when asked *"how do you know they existed?"*